# 01 — Des planches source au dataset Monster

**Objectif :** comprendre l'identité de chaque sprite, son alpha, sa légende, son split et sa provenance.

~~~text
PNG originaux → découpage → identité / forme / vue / palette → alpha
              → centrage 128 × 128 → légende et définition → split par espèce → audit
~~~

Les images 256/512 sont des agrandissements entiers nearest-neighbor des cibles 128. Les sources de 64/96 pixels ne gagnent pas de détails natifs lors de cet agrandissement. Voir le [protocole détaillé](../02-sprite-lora/dataset/README.md).

In [ ]:
from pathlib import Path
import sys

book_dir = next(
    folder
    for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent / "notebooks", parent / "2-LoRA-qwen/notebooks", parent)
    if (folder / "notebook_utils.py").is_file()
)
if str(book_dir) not in sys.path:
    sys.path.insert(0, str(book_dir))
from notebook_utils import (
    PROJECT, REPO, ARTIFACTS, read_json, read_jsonl, sha256,
    contained, current_dataset, archived_report, result_file,
    show_json, table, show_image, run_script,
)
print("Projet :", PROJECT.name)


In [ ]:
dataset, pointer = current_dataset()
info = show_json(dataset / "dataset.json")
audit = read_json(dataset / "audit.json", {})
table([{"contrôle": k, "valeur": audit.get(k)} for k in (
    "passed", "unique_images_checked", "png_files_checked",
    "source_cells_accounted", "train_species", "validation_species"
)], ["contrôle", "valeur"])

## Empreintes et exemples annotés

Le manifeste décrit les noms, formes, genres, styles Emerald/Black-White, vues, palettes, boîtes, correspondances et sources. Les doublons de pixels partagent une image et conservent leurs variantes et provenances.

Les définitions taxonomiques sont distinguées des annotations visibles. Les légendes n'inventent pas de couleur normale pour une variante shiny ni de description anatomique non vérifiée.

In [ ]:
manifest_path = dataset / "manifest.jsonl"
print("SHA-256 attendu :", pointer["manifest_sha256"])
print("SHA-256 lu      :", sha256(manifest_path))
rows = read_jsonl(manifest_path)
table([{
    "id": row["sample_id"], "espèce": row["species"],
    "split": row["split"], "légende": row["text"]
} for row in rows[:6]], ["id", "espèce", "split", "légende"])
species_id = str(rows[0]["species_id"]) if rows else "1"
definition = read_json(dataset / "definitions.json", {}).get(species_id)
print(__import__("json").dumps(definition, ensure_ascii=False, indent=2))

In [ ]:
for row in rows[:4]:
    show_image(contained(dataset, row["file"]), row["sample_id"])

## Séparation par espèce

Toutes les formes, palettes, poses et vues d'une espèce restent dans le même split. Une vue de dos en validation ne doit pas avoir sa vue de face de la même espèce en entraînement. Le contrôle de données complet reste dans `audit_monster_dataset.py`.

In [ ]:
train_species = {str(r["species_id"]) for r in rows if r["split"] == "train"}
validation_species = {str(r["species_id"]) for r in rows if r["split"] == "validation"}
table([
    {"ensemble": "train", "images": info["by_split"]["train"], "espèces": len(train_species)},
    {"ensemble": "validation", "images": info["by_split"]["validation"], "espèces": len(validation_species)},
], ["ensemble", "images", "espèces"])
print("Espèces présentes dans les deux ensembles :", sorted(train_species & validation_species))

## Reproduire la préparation, explicitement

Les références sont téléchargées à leurs révisions figées. Une nouvelle préparation reçoit un nouveau dossier ; la version validée existante conserve ses octets et ses empreintes. Les trois actions sont indépendantes et désactivées par défaut.

In [ ]:
RUN_REFERENCE_DOWNLOAD = False
RUN_NEW_DATASET = False
RUN_DATA_AUDIT = False
new_dataset = PROJECT / "02-sprite-lora/dataset/prepared/monster-clean-v3"

run_script("02-sprite-lora/fetch_monster_references.py", enabled=RUN_REFERENCE_DOWNLOAD)
run_script("02-sprite-lora/prepare_monster_dataset.py",
           ["--output-dir", new_dataset], enabled=RUN_NEW_DATASET)
run_script("02-sprite-lora/audit_monster_dataset.py",
           [new_dataset if RUN_NEW_DATASET else dataset], enabled=RUN_DATA_AUDIT)

La galerie complète du dataset se trouve dans `dataset/prepared/<version>/review/index.html`. Son inspection par agent est identifiée comme telle et ne constitue pas une évaluation humaine statistique du modèle.

**Suite :** [02 — Encodeurs et MSE](02_encodeurs_et_mse.ipynb).